# Temporal vs Spatial: iterations × sparsity cross

**Key question**: If we give the model more iterations (ticks), can extreme sparsity (10% active) catch up to dense?  
If yes → CTM's computation is carried by **temporal depth** (iterations), not **spatial width** (neuron count).

**Δ from baseline** (two parameters):
```
+ iterations    = N            # number of ticks per forward pass
+ topk_neurons  = ratio        # only top-ratio neurons fire per tick
```

In [ ]:
import sys; sys.path.insert(0, '.'); sys.path.insert(0, '..')
from exp_runner import *
import matplotlib.pyplot as plt
import numpy as np
%matplotlib inline

## Experiment design

| Axis | Levels |
|---|---|
| tasks | sort, mazes |
| iterations | 25, 50, 75, 100 |
| topk_neurons | 1.0 (dense), 0.5 (moderate), 0.1 (extreme) |
| seeds | 0, 1, 2 |

Total: 2 × 4 × 3 × 3 = **72 runs**.

In [ ]:
import itertools

iters = [25, 50, 75, 100]
spars = [0.1, 0.5, 1.0]
seeds = range(3)

exps = []
for task in ['sort', 'mazes']:
    module, base = BASE_CONFIGS[task]
    for it, spar, s in itertools.product(iters, spars, seeds):
        name = f'{task}_it{it}_spar{str(spar).replace(".","p")}_s{s}'
        cfg = {**base, 'seed': s, 'iterations': it, 'topk_neurons': spar}
        exps.append(Experiment(name, task, module, cfg))
print(f'Total: {len(exps)} experiments')

In [ ]:
run_all(exps, gpus=8, log_root='logs/deep/05_temporal', dry_run=True)

In [ ]:
done, failed = run_all(exps, gpus=8, log_root='logs/deep/05_temporal')

In [ ]:
status('logs/deep/05_temporal')

## Analysis

In [ ]:
df = collect('logs/deep/05_temporal')
if df.empty:
    df = collect_csv('csv_data/05_results.csv')
    if df.empty:
        print('No results yet.')

if not df.empty:
    df['iterations'] = df.name.str.extract(r'_it(\d+)_')[0].astype(int)
    spar_map = {'0p1': 0.1, '0p5': 0.5, '1p0': 1.0}
    df['sparsity'] = df.name.str.extract(r'_spar(\d+p?\d*)_')[0].map(spar_map)

    for task in ['sort', 'mazes']:
        sub = df[df.task == task].dropna(subset=['best_acc']).copy()
        if sub.empty:
            continue
        bl = BASELINE_ACC[task]

        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

        # --- heatmap: rows=sparsity, cols=iterations, fill=mean best_acc ---
        xv = sorted(sub.iterations.unique())
        yv = sorted(sub.sparsity.unique())
        mat = np.full((len(yv), len(xv)), np.nan)
        for i, y in enumerate(yv):
            for j, x in enumerate(xv):
                vals = sub[(sub.sparsity == y) & (sub.iterations == x)].best_acc
                if not vals.empty:
                    mat[i, j] = vals.mean()
        im = ax1.imshow(mat, cmap='RdYlGn', aspect='auto')
        ax1.set_xticks(range(len(xv)))
        ax1.set_xticklabels(xv, fontsize=10)
        ax1.set_yticks(range(len(yv)))
        ax1.set_yticklabels([f'{y}' for y in yv], fontsize=10)
        ax1.set_xlabel('iterations', fontsize=11)
        ax1.set_ylabel('topk sparsity', fontsize=11)
        ax1.set_title(f'{task}: mean best_acc', fontsize=12, fontweight='bold')
        for i in range(len(yv)):
            for j in range(len(xv)):
                if not np.isnan(mat[i, j]):
                    c = 'white' if abs(mat[i, j] - bl) > 0.1 else 'black'
                    ax1.text(j, i, f'{mat[i, j]*100:.1f}',
                             ha='center', va='center', fontsize=9, fontweight='bold', color=c)
        plt.colorbar(im, ax=ax1, shrink=0.8)

        # --- line plot: acc vs iterations, one line per sparsity ---
        colors = {0.1: '#d62728', 0.5: '#ff7f0e', 1.0: '#2ca02c'}
        for spar in sorted(sub.sparsity.unique()):
            sg = sub[sub.sparsity == spar].groupby('iterations').best_acc
            means = sg.mean()
            stds = sg.std(ddof=1)
            ax2.errorbar(means.index, means.values * 100,
                         yerr=stds.values * 100 if not stds.isna().all() else None,
                         fmt='-o', color=colors.get(spar, '#888'), capsize=4,
                         label=f'sparsity={spar}', linewidth=2, markersize=7)
        ax2.axhline(bl * 100, color='gray', ls='--', alpha=0.5,
                    label=f'no-sparsity baseline ({bl*100:.1f}%)')
        ax2.set_xlabel('iterations', fontsize=11)
        ax2.set_ylabel('best test acc (%)', fontsize=11)
        ax2.set_title(f'{task}: iterations × sparsity', fontsize=12, fontweight='bold')
        ax2.legend(fontsize=9)
        ax2.grid(True, alpha=0.2)

        fig.tight_layout()
        fig.savefig(f'figures/05_{task}_tempspar.png', dpi=150, bbox_inches='tight')
        plt.show()